In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pmdarima import auto_arima
from sklearn.metrics import mean_squared_error


In [3]:
df = pd.read_csv("LahoreDivision_Tehsil_Rainfall_1995_2025.csv")

df["date"] = pd.to_datetime(dict(year=df["year"], month=df["month"], day=1))
df = df.sort_values("date")


In [5]:
results = {}

for tehsil in df["adm3_name"].unique():
    
    data = df[df["adm3_name"] == tehsil]
    ts = data.set_index("date")["rainfall_mm"]
    
    train_size = int(len(ts) * 0.8)
    train, test = ts[:train_size], ts[train_size:]
    
    model = auto_arima(
        train,
        seasonal=True,
        m=12,
        stepwise=True,
        suppress_warnings=True,
        trace=False
    )
    
    forecast = model.predict(n_periods=len(test))
    
    rmse = np.sqrt(mean_squared_error(test, forecast))
    
    results[tehsil] = {
        "model": model,
        "rmse": rmse,
        "test": test,
        "forecast": forecast
    }
    
    print(tehsil, "RMSE:", rmse)


Chunian RMSE: 39.7858278569023
Kasur RMSE: 44.43320077948973
Pattoki RMSE: 47.60532023729733
Lahore Cantt RMSE: 58.869015227527626
Lahore City RMSE: 54.99892082890488
Nankana Sahib RMSE: 47.150574035493186
Sangla Hill RMSE: 36.54451832201527
Ferozewala RMSE: 54.74110076522196
Muridke RMSE: 55.90903287980054
Safdarabad RMSE: 38.68143892184063
Sheikhupura RMSE: 43.39910509521578


In [14]:
future_forecasts = {}

for tehsil in results:
    model = results[tehsil]["model"]
    future_values = model.predict(n_periods=12)

    future_dates = pd.date_range(start="2026-01-01", periods=12, freq='MS')
    future_df = pd.DataFrame({
        "date": future_dates,
        "predicted_rainfall_mm": future_values
    })
    
    future_forecasts[tehsil] = future_df
    
    print("\nFuture rainfall for", tehsil)
    print(future_df)



Future rainfall for Chunian
                 date  predicted_rainfall_mm
2019-10-01 2026-01-01               4.025768
2019-11-01 2026-02-01               2.837660
2019-12-01 2026-03-01               3.600837
2020-01-01 2026-04-01              16.873893
2020-02-01 2026-05-01              22.092588
2020-03-01 2026-06-01              19.513045
2020-04-01 2026-07-01              18.765767
2020-05-01 2026-08-01              13.061136
2020-06-01 2026-09-01              49.494308
2020-07-01 2026-10-01             104.168789
2020-08-01 2026-11-01              76.134354
2020-09-01 2026-12-01              51.556140

Future rainfall for Kasur
                 date  predicted_rainfall_mm
2019-10-01 2026-01-01               4.367509
2019-11-01 2026-02-01               3.205122
2019-12-01 2026-03-01               4.608812
2020-01-01 2026-04-01              19.844039
2020-02-01 2026-05-01              26.558016
2020-03-01 2026-06-01              23.818765
2020-04-01 2026-07-01              19.133925